In [2]:
import pandas as pd
import sqlite3

# Load the cleaned data from Phase 4
runs = pd.read_csv("../data/processed/production_runs_clean.csv")
sensors = pd.read_csv("../data/processed/sensor_readings_clean.csv")
defects = pd.read_csv("../data/processed/defect_events_clean.csv")
machines = pd.read_csv("../data/processed/machines_clean.csv")

print("Runs:", runs.shape)
print("Sensors:", sensors.shape)
print("Defects:", defects.shape)
print("Machines:", machines.shape)

# Connect to (or create) the SQLite database file
conn = sqlite3.connect("../database/yieldwatch.db")
print("\nConnected to database.")

Runs: (60000, 13)
Sensors: (60000, 11)
Defects: (59959, 5)
Machines: (20, 5)

Connected to database.


In [3]:
# Load each table into the database
runs.to_sql("production_runs", conn, if_exists="replace", index=False)
sensors.to_sql("sensor_readings", conn, if_exists="replace", index=False)
defects.to_sql("defect_events", conn, if_exists="replace", index=False)
machines.to_sql("machines", conn, if_exists="replace", index=False)

conn.commit()
print("All 4 tables loaded into yieldwatch.db")

All 4 tables loaded into yieldwatch.db


In [4]:
# Verify: ask the database directly what tables exist
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

# Quick row count check via SQL (not Pandas) to confirm the data really landed
for table in ["production_runs", "sensor_readings", "defect_events", "machines"]:
    count = pd.read_sql(f"SELECT COUNT(*) as row_count FROM {table}", conn)
    print(f"{table}: {count['row_count'][0]} rows")

              name
0  production_runs
1  sensor_readings
2    defect_events
3         machines
production_runs: 60000 rows
sensor_readings: 60000 rows
defect_events: 59959 rows
machines: 20 rows


In [5]:
# Prove production_runs and sensor_readings are properly linked via run_id
join_check = pd.read_sql("""
    SELECT r.run_id, r.machine_id, r.shift_id, s.temperature, s.vibration
    FROM production_runs r
    JOIN sensor_readings s ON r.run_id = s.run_id
    LIMIT 5
""", conn)

print(join_check)

# Also confirm every sensor_reading actually points to a real run (no orphaned records)
orphan_check = pd.read_sql("""
    SELECT COUNT(*) as orphan_sensors
    FROM sensor_readings s
    LEFT JOIN production_runs r ON s.run_id = r.run_id
    WHERE r.run_id IS NULL
""", conn)

print("\nOrphaned sensor readings (should be 0):", orphan_check["orphan_sensors"][0])

    run_id machine_id shift_id  temperature  vibration
0  R000001        M01  Morning        79.50       5.25
1  R000002        M20  Evening        73.17       2.76
2  R000003        M11    Night        70.81       5.33
3  R000004        M01    Night        82.39       6.54
4  R000005        M06  Evening        71.73       5.33

Orphaned sensor readings (should be 0): 0


In [6]:
conn.close()
print("Database connection closed.")

Database connection closed.
